# Data Cleaning for Portfolio Optimization

This notebook cleans the BEL20 stock data and prepares it for the portfolio optimization solver.

In [ ]:
import pandas as pd
import numpy as np
import json

## 1. Loading Raw Data

In [6]:
input = pd.read_excel("../../data/init_all.xlsx")

## 2. Converting Raw Data into More Usable Format

In [ ]:
df_vals = input.set_index('Ticker')

# Excel dates are automatically converted by pandas, no conversion needed
df_vals = df_vals.transpose()

# Drop columns with any NA values
df_vals = df_vals.dropna(axis=1)

# Convert all data to float
df_vals = df_vals.astype(float)

print("############# New data format: #######")
print(df_vals.info())

df_vals.head()

df_vals.info()

In [ ]:
print("=== Checking for negative values ===")
has_negatives = (df_vals < 0).any().any()
print(f"Any negative values found: {has_negatives}")

if has_negatives:
    print("\nTickers with negative values:")
    for ticker in df_vals.columns:
        negatives = (df_vals[ticker] < 0).sum()
        if negatives > 0:
            print(f"  {ticker}: {negatives} negative values (min: {df_vals[ticker].min()})")
else:
    print("✓ No negative values - data quality check passed")

In [ ]:
# Remove any columns (stocks) that contain negative values
print("=== Removing stocks with negative values ===")
cols_before = df_vals.shape[1]

# Identify columns with any negative values
cols_to_keep = ~(df_vals < 0).any()
df_vals = df_vals.loc[:, cols_to_keep]

cols_after = df_vals.shape[1]
cols_removed = cols_before - cols_after

if cols_removed > 0:
    print(f"Removed {cols_removed} stocks with negative values")
    print(f"Remaining stocks: {cols_after}")
else:
    print(f"✓ No stocks removed - all {cols_after} stocks are clean")

In [ ]:
import json

# Load currency mappings and exchange rates
with open("../../data/ticker_currencies.json") as f:
    ticker_currencies = json.load(f)

with open("../../data/exchange_rates.json") as f:
    exchange_rates = json.load(f)

# Build lookup: ticker -> currency
currency_map = {ticker: ticker_currencies[ticker] for ticker in df_vals.columns if ticker in ticker_currencies}

# Conversion counters
eur_passthrough = 0
conversions_applied = 0
stocks_converted = set()

print("=== Currency Conversion to EUR ===")

# Loop through each date (row) and apply conversions
for date in df_vals.index:
    date_str = date.strftime('%Y-%m-%d')
    
    if date_str not in exchange_rates:
        raise KeyError(f"Missing exchange rates for date: {date_str}")
    
    daily_rates = exchange_rates[date_str]
    
    # Convert each stock
    for ticker in df_vals.columns:
        if ticker not in currency_map:
            continue
        
        currency = currency_map[ticker]
        stocks_converted.add(ticker)
        
        if currency == "EUR":
            eur_passthrough += 1
        else:
            if currency not in daily_rates:
                raise KeyError(f"Missing exchange rate for {currency} on {date_str}")
            
            rate = daily_rates[currency]
            df_vals.loc[date, ticker] *= rate
            conversions_applied += 1

print(f"✓ Conversion complete:")
print(f"  - Stocks processed: {len(stocks_converted)}")
print(f"  - EUR pass-throughs: {eur_passthrough}")
print(f"  - Non-EUR conversions applied: {conversions_applied}")

## 2.3 Currency Conversion to EUR

## 2.2 Data Quality Check: Negative Values

## 3. Computing Average Geometric Returns

We use a simple (readable) hand built python function for this

Note: We see that we are missing 125 entries for SYENS.

In [8]:
def avg_geo_return(time_series_list : list, test : bool = False):
    """ Takes a list of values, and calculates the 
     average geometric change per unit of time (assuming each entry
     is one unit if time).
      
    See the latex in the prelim report for this logic written out in
     a readable manner for non pyton users """
    starting_value = time_series_list[0]
    ending_value = time_series_list[-1]
    nr_time_period = len(time_series_list) -1 #With N entries there are N -1 time periods
    increase_multiplier = ending_value / starting_value
    avg_geo_change = increase_multiplier**(1/nr_time_period)

    if test == True:
        print("Total increase multiplier")
        print(increase_multiplier)
        print("Average geo change per time period")
        print(avg_geo_change)

    return avg_geo_change

### Testing the function with First ticker

In [ ]:
print("####### Testing for first ticker ##########")
first_ticker = df_vals.columns[0]
print(f"Testing with ticker: {first_ticker}")
ticker_list = df_vals[first_ticker].tolist()
avg_geo_return(ticker_list, test=True)

### Computing avg geo return for all tickers

In [10]:
exp_ret_list = list()

for ticker in df_vals.columns:
    ticker_values = df_vals[ticker].dropna().to_list() 
    #For a given ticker, we drop NA values. See the report for a discussion of the 
    #implications of this
    exp_ret_list.append(avg_geo_return(ticker_values))
#Note that computing it like this gives us the avg geo return over the entire time horizon

TypeError: unsupported operand type(s) for /: 'float' and 'str'

## 4. Computing Covariance Matrix

Up until now we have been working with the absolute values of the stocks.

For our analysis, this is not an interesting metric. We are interested in the percentage change of the value of the stock

In [ ]:
df_delta = df_vals.pct_change()

print("############ Testing for pct change transformation #######")
print("Delta")
print(df_delta.head(n=3))
print("Absolute vals")
print(df_vals.head(n=3))
print("Just take a calculator and run the numbers")

### Computing the covariance matrix

In [ ]:
print('############ covariance matrix ########')

cov_matrix = df_delta.cov() 
""" This computes the pairvise (sample) conveariance, this excludes all NA values from
#the computation. Which is the behaviour we want. So entries where SYENS = NA is not considered
Since the computations are pairwise the NA values for SYENS do not affecf cov pairs that SYENS is not a part of

The documenation explains this function: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.cov.html#pandas.DataFrame.cov
"""

print(cov_matrix)

## 5. Writing Cleaned Data to JSON

In [ ]:
data_cleaned = {
    "tickers": df_vals.columns.to_list(),
    "expected_returns": exp_ret_list,
    "covariance_matrix": cov_matrix.values.tolist()
}

output_path = "../../data/expected_returns_covariance_matrix.json"
with open(output_path, "w") as f:
    json.dump(data_cleaned, f)

print(f"Data cleaning complete. Output saved to {output_path}")